In [ ]:
#pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [28]:
import pandas as pd
import sqlalchemy
import pymysql

In [44]:
schema = "liane_library"
host = "127.0.0.1"
user = "root"
password = "Vihan2014("
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [45]:
pd.read_sql('books', con=connection_string)

,title,author,genre,published_year,isbn
0,Clean Code,Robert C. Martin,Programming,2008,9780132350884
1,Der kleine Prinz,Antoine de Saint-Exupéry,Fantasy,1943,9780156012195
2,Harry Potter und der Stein der Weisen,J.K. Rowling,Fantasy,1997,9783551354013
3,Die Verwandlung,Franz Kafka,Literatur,1915,9783596900012


# Create

## Create Book

In [60]:
from sqlalchemy import create_engine, text

# Create an engine to connect to the database
engine = create_engine(connection_string)

# Define the update query
update_query = """
    INSERT INTO books (title, author, isbn)
VALUES ('test buch', 'test author', '1234567891013');
"""

# Execute the update query with an explicit commit
with engine.connect() as connection:
    transaction = connection.begin()
    try:
        connection.execute(text(update_query))
        transaction.commit()
    except:
        transaction.rollback()
        raise

# Read

In [61]:
pd.read_sql("""
    SELECT
    books.title,
    loans.loan_states,
    friends.friend_name,
    friends.friend_id
FROM books
JOIN loans
    ON books.isbn = loans.book_id
JOIN friends
    ON loans.friend_id = friends.friend_id;


;
""", con=connection_string)


,title,loan_states,friend_name,friend_id
0,Der kleine Prinz,returned,Anna,1
1,Harry Potter und der Stein der Weisen,loaned,Markus,2
2,Die Verwandlung,delayed,Sophie,3
3,Clean Code,missing,Lukas,4


UPDATE

In [55]:

from sqlalchemy import create_engine, text

# Create an engine to connect to the database
engine = create_engine(connection_string)

# Define the update query
update_query = """
    UPDATE loans 
SET return_date = '2026-07-09'
WHERE book_id = '9780156012195' and friend_id = '1';
"""

# Execute the update query with an explicit commit
with engine.connect() as connection:
    transaction = connection.begin()
    try:
        connection.execute(text(update_query))
        transaction.commit()
    except:
        transaction.rollback()
        raise

DELETE

In [63]:
from sqlalchemy import create_engine, text

# Create an engine to connect to the database
engine = create_engine(connection_string)

# Define the update query
update_query = """
   DELETE FROM books
WHERE isbn = '9783551354013';
"""

# Execute the update query with an explicit commit
with engine.connect() as connection:
    transaction = connection.begin()
    try:
        connection.execute(text(update_query))
        transaction.commit()
    except:
        transaction.rollback()
        raise

In [65]:
pd.read_sql('books', con = connection_string)

,title,author,genre,published_year,isbn
0,Clean Code,Robert C. Martin,Programming,2008,9780132350884
1,Der kleine Prinz,Antoine de Saint-Exupéry,Fantasy,1943,9780156012195
2,Die Verwandlung,Franz Kafka,Literatur,1915,9783596900012


In [ ]:
pd.read_sql('''select * from books 
            inner join loans 
            where loan_states = "delayed"''', con = connection_string)

,title,author,genre,published_year,isbn,book_id,friend_id,loan_date,return_date,last_contact,next_contact,note,loan_states
0,Clean Code,Robert C. Martin,Programming,2008,9780132350884,9780156012195,1,2026-01-10,2026-01-20,2026-01-15,None,Buch wurde zurückgegeben,returned
1,Der kleine Prinz,Antoine de Saint-Exupéry,Fantasy,1943,9780156012195,9780156012195,1,2026-01-10,2026-01-20,2026-01-15,None,Buch wurde zurückgegeben,returned
2,Die Verwandlung,Franz Kafka,Literatur,1915,9783596900012,9780156012195,1,2026-01-10,2026-01-20,2026-01-15,None,Buch wurde zurückgegeben,returned


In [71]:
def test(x,y):
    return x+y
        

In [72]:
test(3,2)

5

In [73]:
def delayed_books():
    df = pd.read_sql('''select * from books 
                inner join loans 
                where loan_states = "delayed"''', con = connection_string)
    return df

In [74]:
delayed_books()

,title,author,genre,published_year,isbn,book_id,friend_id,loan_date,return_date,last_contact,next_contact,note,loan_states
0,Clean Code,Robert C. Martin,Programming,2008,9780132350884,9783596900012,3,2026-01-05,None,2026-02-01,2026-02-15,Mehrfach erinnert,delayed
1,Der kleine Prinz,Antoine de Saint-Exupéry,Fantasy,1943,9780156012195,9783596900012,3,2026-01-05,None,2026-02-01,2026-02-15,Mehrfach erinnert,delayed
2,Die Verwandlung,Franz Kafka,Literatur,1915,9783596900012,9783596900012,3,2026-01-05,None,2026-02-01,2026-02-15,Mehrfach erinnert,delayed


In [78]:
def delayed_books():
    df = pd.read_sql('''select books.title, friends.friend_name, loans.book_id, loans.due_date from books
                join loans 
                ON books.isbn = loans.book_id
                join friends
                ON loans.friend_id = friends.friend_id
                where loans.loan_states = "delayed"''', con = connection_string)
    return df

In [79]:
delayed_books()

,title,friend_name,book_id,due_date
0,Die Verwandlung,Sophie,9783596900012,2026-01-19


In [86]:
def update_return_date(isbn, friend_id, return_date):    
    engine = create_engine(connection_string)

    # Define the update query
    update_query = f"""UPDATE loans 
        SET return_date = '{return_date}'
        WHERE book_id = {isbn} and friend_id = {friend_id};"""

    # Execute the update query with an explicit commit
    with engine.connect() as connection:
        transaction = connection.begin()
        try:
            connection.execute(text(update_query))
            transaction.commit()
        except:
            transaction.rollback()
            raise
    


In [87]:
update_return_date(9783596900012, 3, '2026-10-17')

In [88]:
pd.read_sql('loans',con=connection_string)

,book_id,friend_id,loan_date,return_date,due_date,last_contact,next_contact,note,loan_states
0,9780156012195,1,2026-01-10,2026-07-09,2026-01-17,2026-01-15,NaT,Buch wurde zurückgegeben,returned
1,9783551354013,2,2026-02-01,NaT,2026-02-15,2026-02-10,2026-02-20,Freund liest noch,loaned
2,9783596900012,3,2026-01-05,2026-10-17,2026-01-19,2026-02-01,2026-02-15,Mehrfach erinnert,delayed
3,9780132350884,4,2025-12-01,NaT,2025-12-15,2026-01-10,2026-02-01,Buch nicht auffindbar,missing


In [90]:
def update_notes(friend_id, notiz):    
    engine = create_engine(connection_string)

    # Define the update query
    update_query = f"""UPDATE friends 
        SET notes = '{notiz}'
        WHERE friend_id = {friend_id};"""

    # Execute the update query with an explicit commit
    with engine.connect() as connection:
        transaction = connection.begin()
        try:
            connection.execute(text(update_query))
            transaction.commit()
        except:
            transaction.rollback()
            raise
    

In [91]:
update_notes(3, 'hallo wie geht es ')

In [92]:
pd.read_sql('friends', con=connection_string)

,friend_name,max_loans,notes,friend_id
0,Anna,3,Mag Fantasy Bücher,1
1,Markus,2,Liest hauptsächlich Programmierbücher,2
2,Sophie,4,hallo wie geht es,3
3,Lukas,1,Gibt Bücher schnell zurück,4


In [93]:
def update_loan_states(friend_id, book_id, status):
    engine = create_engine(connection_string)
    
        # Define the update query
    update_query = f"""UPDATE loans 
            SET loan_states = '{status}'
            WHERE friend_id = {friend_id} AND book_id={'book_id'};"""
    
    # Execute the update query with an explicit commit
    with engine.connect() as connection:
        transaction = connection.begin()
        try:
            connection.execute(text(update_query))
            transaction.commit()
        except:
                transaction.rollback()
                raise

In [95]:
update_loan_states(4, 9780132350884, 'Returned')

In [96]:
pd.read_sql('loans', con=connection_string)

,book_id,friend_id,loan_date,return_date,due_date,last_contact,next_contact,note,loan_states
0,9780156012195,1,2026-01-10,2026-07-09,2026-01-17,2026-01-15,NaT,Buch wurde zurückgegeben,returned
1,9783551354013,2,2026-02-01,NaT,2026-02-15,2026-02-10,2026-02-20,Freund liest noch,loaned
2,9783596900012,3,2026-01-05,2026-10-17,2026-01-19,2026-02-01,2026-02-15,Mehrfach erinnert,delayed
3,9780132350884,4,2025-12-01,NaT,2025-12-15,2026-01-10,2026-02-01,Buch nicht auffindbar,Returned


In [97]:
def update_due_date(friend_id, book_id, due_date):
    engine = create_engine(connection_string)
    
        # Define the update query
    update_query = f"""UPDATE loans 
            SET due_date = '{due_date}'
            WHERE friend_id = {friend_id} AND book_id='{book_id}';"""
    
    # Execute the update query with an explicit commit
    with engine.connect() as connection:
        transaction = connection.begin()
        try:
            connection.execute(text(update_query))
            transaction.commit()
        except:
                transaction.rollback()
                raise

In [99]:
update_due_date(4, 9780132350884, '2025-12-25')

In [100]:
pd.read_sql('loans', con=connection_string)

,book_id,friend_id,loan_date,return_date,due_date,last_contact,next_contact,note,loan_states
0,9780156012195,1,2026-01-10,2026-07-09,2026-01-17,2026-01-15,NaT,Buch wurde zurückgegeben,returned
1,9783551354013,2,2026-02-01,NaT,2026-02-15,2026-02-10,2026-02-20,Freund liest noch,loaned
2,9783596900012,3,2026-01-05,2026-10-17,2026-01-19,2026-02-01,2026-02-15,Mehrfach erinnert,delayed
3,9780132350884,4,2025-12-01,NaT,2025-12-25,2026-01-10,2026-02-01,Buch nicht auffindbar,missing


In [ ]:
def get_friend_id(friend_name):
    query = """
        SELECT friend_id
        FROM friends
        WHERE friend_name = %(name)s
    """

    df = pd.read_sql(
        query,
        con=connection_string,
        params={"name": friend_name}
    )

    if df.empty:
        return None

    return df.iloc[0]["friend_id"]

In [103]:
df = pd.read_sql("""
        SELECT friend_id
        FROM friends
        WHERE friend_name = 'Anna'
    """, con=connection_string)

In [104]:
type(df)

pandas.core.frame.DataFrame

In [107]:
df.iloc[0,0]

np.int64(1)

In [ ]:
#Liste alle freunde aus mysql
[Anna, Markus, Sophie, Lukas]
#drop down mit diese liste
#name für friend id benutzen
#friend_id für update note benutzen

In [135]:
 friends_list = pd.read_sql("""
        SELECT friend_name
        FROM friends
    """, con=connection_string)["friend_name"]

In [136]:
friends_list

0      Anna
1    Markus
2    Sophie
3     Lukas
Name: friend_name, dtype: object

In [ ]:
def all_friends():
    friends_list = pd.read_sql("""
        SELECT friend_name
        FROM friends
    """, con=connection_string)["friend_name"] 
    return friends_list

In [ ]:
### für die select box , SELECT title FROM Books

In [ ]:
def get_book_id(book_title):
    query = """
        SELECT isbn
        FROM books
        WHERE title = %(title)s
    """

    df = pd.read_sql(
        query,
        con=connection_string,
        params={"title": book_title}
    )

    if df.empty:
        return None

    return df.iloc[0]["isbn"]

In [123]:
df = pd.read_sql( """
        SELECT isbn
        FROM books
        WHERE title = "Clean Code"
    """, con=connection_string)

In [124]:
type(df)

pandas.core.frame.DataFrame

In [125]:
df.iloc[0,0]

'9780132350884'

In [138]:
books_title_list = pd.read_sql( """
        SELECT title
        FROM books
    """, con=connection_string)["title"]

In [139]:
books_title_list

0                               Clean Code
1                         Der kleine Prinz
2    Harry Potter und der Stein der Weisen
3                          Die Verwandlung
Name: title, dtype: object

In [137]:
def all_titles():
    books_title_list = pd.read_sql( """
        SELECT title
        FROM books
    """, con=connection_string)["title"]
    return books_title_list
